In [22]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side

# ================= CONFIG =================
INPUT_FILE = "/Users/maryam/Downloads/Property Equipment and Energy Survey.xlsx"
OUTPUT_FILE = "/Users/maryam/Downloads/Property_Survey_Template.xlsx"

PROPERTY_NAMES = [
    "Citadines Les Halles",
    "Citadines Saint Germain",
    "Citadines XXXX"
]
# =========================================

# -------- READ INPUT --------
df = pd.read_excel(INPUT_FILE, usecols=[0], header=None)
df.columns = ["text"]
df["text"] = df["text"].astype(str).str.strip()

sections = {}
current_section = None
current_question = None

# -------- PARSE STRUCTURE --------
for line in df["text"]:
    if not line or line.lower() == "nan":
        continue

    # SECTION headers
    if line.upper().startswith("SECTION"):
        current_section = line
        sections[current_section] = []
        current_question = None
        continue

    # Numbered question (e.g., 1.1., 2.1.)
    if line[0].isdigit() and "." in line:
        current_question = line
        sections[current_section].append(current_question)
        continue

    # Lettered sub-question (e.g., a. or b. with or without space)
    if len(line) >= 2 and line[0].isalpha() and line[1] == ".":
        current_question = line
        sections[current_section].append(current_question)
        continue

    # Multi-line continuation of previous question
    if current_question:
        sections[current_section][-1] += " " + line

# -------- WRITE EXCEL --------
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for section, questions in sections.items():
        columns = ["Properties"] + questions
        df_out = pd.DataFrame(columns=columns)
        for prop in PROPERTY_NAMES:
            df_out.loc[len(df_out)] = [prop] + [""] * len(questions)

        # Safe sheet name
        sheet_name = section.split("—")[0].replace("SECTION", "Section").strip()[:31]
        df_out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=2)

# -------- FORMAT EXCEL --------
wb = load_workbook(OUTPUT_FILE)
thin = Side(style="thin")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for section, questions in sections.items():
    sheet_name = section.split("—")[0].replace("SECTION", "Section").strip()[:31]
    ws = wb[sheet_name]

    total_cols = len(questions) + 1
    total_rows = ws.max_row

    # Merge SECTION header
    ws.merge_cells(start_row=1, start_column=2,
                   end_row=1, end_column=total_cols)
    ws.cell(row=1, column=2).value = section
    ws.cell(row=1, column=2).font = Font(bold=True)
    ws.cell(row=1, column=2).alignment = Alignment(horizontal="center")

    # Header formatting
    for c in range(1, total_cols + 1):
        ws.cell(row=2, column=c).font = Font(bold=True)
        ws.cell(row=2, column=c).alignment = Alignment(
            wrap_text=True, horizontal="center", vertical="center"
        )
        ws.column_dimensions[chr(64 + c)].width = 28

    ws.column_dimensions["A"].width = 35

    # Borders
    for r in range(2, total_rows + 1):
        for c in range(1, total_cols + 1):
            ws.cell(row=r, column=c).border = border

    # Freeze headers + Properties column
    ws.freeze_panes = "B3"

wb.save(OUTPUT_FILE)



In [23]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side

# ================= CONFIG =================
INPUT_FILE = "/Users/maryam/Downloads/Property Equipment and Energy Survey.xlsx"
OUTPUT_FILE = "/Users/maryam/Downloads/Property_Survey_Template2.xlsx"

PROPERTY_NAMES = [
    "Citadines Danube Vienna",
"Citadines Toison d'Or Brussels",
"Citadines Sainte-Catherine Brussels",
"Citadines Saint-Germain-des-Prés Paris",
"La Clef Tour Eiffel Paris by The Crest Collection",
"La Clef Champs-Élysées Paris by The Crest Collection",
"Citadines Les Halles Paris",
"Citadines Opéra Paris",
"Citadines Trocadéro Paris",
"Citadines Bastille Marais Paris",
"Citadines Tour Eiffel Paris",
"La Clef Louvre Paris by The Crest Collection",
"Citadines Place d’Italie Paris",
"Citadines Montmartre Paris",
"Citadines Antigone Montpellier",
"Citadines Montparnasse Paris",
"Citadines Bastille Gare de Lyon Paris",
"Citadines Wilson Toulouse",
"Citadines Austerlitz Paris",
"Citadines République Paris",
"Citadines Part-Dieu Lyon",
"Citadines Presqu'île Lyon",
"Citadines Kléber Strasbourg",
"Citadines La Défense Paris",
"lyf Gambetta Paris",
"Citadines City Centre Tbilisi",
"Citadines Arnulfpark Munich",
"Citadines Kurfürstendamm Berlin",
"Citadines City Centre Frankfurt",
"Citadines Michel Hamburg",
"lyf East Frankfurt",
"Temple Bar Hotel Dublin by The Unlimited Collection",
"Citadines Canal Amsterdam",
"Citadines Ramblas Barcelona",
"The Cavendish London",
"Citadines Trafalgar Square London",
"Mount Royal Hotel Edinburgh by The Unlimited Collection",
"Citadines Barbican London",
"Citadines Holborn-Covent Garden London",
"Citadines South Kensington London",
"Citadines Islington London",
"Citadines City Centre Liverpool",
]
# =========================================

# -------- READ INPUT --------
df = pd.read_excel(INPUT_FILE, usecols=[0], header=None)
df.columns = ["text"]
df["text"] = df["text"].astype(str).str.strip()

sections = {}
current_section = None
current_question = None

# -------- PARSE STRUCTURE --------
for line in df["text"]:
    if not line or line.lower() == "nan":
        continue

    # SECTION headers
    if line.upper().startswith("SECTION"):
        current_section = line
        sections[current_section] = []
        current_question = None
        continue

    # Numbered question
    if line[0].isdigit() and "." in line:
        current_question = line
        sections[current_section].append(current_question)
        continue

    # Lettered sub-question
    if len(line) >= 2 and line[0].isalpha() and line[1] == ".":
        current_question = line
        sections[current_section].append(current_question)
        continue

    # Multi-line continuation
    if current_question:
        sections[current_section][-1] += " " + line

# -------- COMBINE ALL QUESTIONS --------
all_questions = []
section_boundaries = []  # track start/end columns for each section

col_idx = 2  # Column B is the first question column
for section, questions in sections.items():
    section_boundaries.append((section, col_idx, col_idx + len(questions) - 1))
    all_questions.extend(questions)
    col_idx += len(questions)

# -------- CREATE DATAFRAME --------
columns = ["Properties"] + all_questions
df_out = pd.DataFrame(columns=columns)

for prop in PROPERTY_NAMES:
    df_out.loc[len(df_out)] = [prop] + [""] * len(all_questions)

df_out.to_excel(OUTPUT_FILE, index=False, startrow=3) 

# -------- FORMAT EXCEL --------
wb = load_workbook(OUTPUT_FILE)
ws = wb.active

thin = Side(style="thin")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

total_cols = len(all_questions) + 1
total_rows = ws.max_row

# Row 1: Big survey title
ws.merge_cells(start_row=1, start_column=2, end_row=1, end_column=total_cols)
ws.cell(row=1, column=2).value = "Property Equipment and Energy Survey"
ws.cell(row=1, column=2).font = Font(bold=True, size=14)
ws.cell(row=1, column=2).alignment = Alignment(horizontal="center")

# Row 2: Section headers
for section, start_col, end_col in section_boundaries:
    ws.merge_cells(start_row=2, start_column=start_col, end_row=2, end_column=end_col)
    ws.cell(row=2, column=start_col).value = section
    ws.cell(row=2, column=start_col).font = Font(bold=True)
    ws.cell(row=2, column=start_col).alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Row 3: Question headers (already written by pandas)
for c in range(1, total_cols + 1):
    ws.cell(row=3, column=c).font = Font(bold=True)
    ws.cell(row=3, column=c).alignment = Alignment(
        wrap_text=True, horizontal="center", vertical="center"
    )

# Column widths
ws.column_dimensions["A"].width = 35
for c in range(2, total_cols + 1):
    ws.column_dimensions[chr(64 + c if c <= 26 else 65 + c-27)].width = 28

# Borders
for r in range(2, total_rows + 1):
    for c in range(1, total_cols + 1):
        ws.cell(row=r, column=c).border = border

# Freeze headers
ws.freeze_panes = "B4"

wb.save(OUTPUT_FILE)
